In [1]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_DIR = Path("/kaggle/input/competitions/industrial-surface-defect-detection/public")
TRAIN_CSV = DATA_DIR / "train.csv"
IMAGE_DIR = DATA_DIR / "train"

train = pd.read_csv(TRAIN_CSV)

print("Device:", DEVICE)
print("Dataset shape:", train.shape)
print("Unique images:", train["image_id"].nunique())
print("Image directory exists:", IMAGE_DIR.exists())
print(train.head())

Device: cpu
Dataset shape: (45572, 5)
Unique images: 11393
Image directory exists: True
     image_id defect_type                                           mask_rle  \
0  4S22I7SA34    defect_a  7255 6 7377 12 7505 12 7633 12 7762 11 7890 11...   
1  4S22I7SA34    defect_b                                                NaN   
2  4S22I7SA34    defect_c                                                NaN   
3  4S22I7SA34    defect_d                                                NaN   
4  DZ899111OE    defect_a                                                NaN   

   width  height  
0    800     128  
1    800     128  
2    800     128  
3    800     128  
4    800     128  


In [2]:
# ==========================================
# Q1: FEATURE MAP SPATIAL GEOMETRY
# ==========================================

# Original image dimensions
H, W = 128, 800

print("Even-width input: ")
for stage in range(1, 5):
    H = H // 2
    W = W // 2
    print(f"Stage {stage}: ({H}, {W})")

H_bottle, W_bottle = H, W

# Odd-width input: 799
odd_H, odd_W = 128, 799

print("\nOdd-width input:")
for stage in range(1, 5):
    odd_H = odd_H // 2
    odd_W = odd_W // 2
    print(f"Stage {stage}: ({odd_H}, {odd_W})")

# Skip connection mismatch
mismatch_px = abs(W_bottle - odd_W)

# Spatial downsampling factor
downsample_factor = (128 * 800) / (H_bottle * W_bottle)

print("\nQ1 FINAL RESULTS")
print("H_bottle:", H_bottle)
print("W_bottle:", W_bottle)
print("Mismatch:", mismatch_px)
print("Downsampling factor:", downsample_factor)

print(
    f"\nQ1 Answer: {H_bottle}, {W_bottle}; "
    f"{mismatch_px}; {downsample_factor:.0f}"
)

Even-width input: 
Stage 1: (64, 400)
Stage 2: (32, 200)
Stage 3: (16, 100)
Stage 4: (8, 50)

Odd-width input:
Stage 1: (64, 399)
Stage 2: (32, 199)
Stage 3: (16, 99)
Stage 4: (8, 49)

Q1 FINAL RESULTS
H_bottle: 8
W_bottle: 50
Mismatch: 1
Downsampling factor: 256.0

Q1 Answer: 8, 50; 1; 256


In [4]:
%pip install -q segmentation-models-pytorch

Note: you may need to restart the kernel to use updated packages.


In [5]:
import segmentation_models_pytorch as smp

print("SMP version:", smp.__version__)
print("Library imported successfully!")

SMP version: 0.6.0
Library imported successfully!


In [6]:
# ==========================================
# Q2: BACKBONE PARAMETER AUDIT
# ==========================================

import segmentation_models_pytorch as smp

# Create both models
model_a = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=1,
    classes=4
)

model_b = smp.Unet(
    encoder_name="efficientnet-b0",
    encoder_weights="imagenet",
    in_channels=1,
    classes=4
)

# Count all parameters
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

params_a = count_parameters(model_a)
params_b = count_parameters(model_b)

print("ResNet34 U-Net parameters:", f"{params_a:,}")
print("EfficientNet-B0 U-Net parameters:", f"{params_b:,}")

# Identify parameter-efficient architecture
if params_a < params_b:
    efficient_model = "resnet34"
else:
    efficient_model = "efficientnet-b0"

param_diff = abs(params_a - params_b)

# Freeze ResNet34 encoder
for p in model_a.encoder.parameters():
    p.requires_grad = False

trainable_params = sum(
    p.numel() for p in model_a.parameters()
    if p.requires_grad
)

trainable_pct = 100 * trainable_params / params_a

print("\nQ2 RESULTS")
print("More parameter-efficient:", efficient_model)
print("Parameter difference:", f"{param_diff:,}")
print("ResNet34 trainable parameters:", f"{trainable_params:,}")
print("ResNet34 trainable percentage:", f"{trainable_pct:.1f}%")

config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 87.3MB            

model.safetensors: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/106 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 21.4MB            

model.safetensors: downloading bytes:           |  0.00B            

ResNet34 U-Net parameters: 24,430,532
EfficientNet-B0 U-Net parameters: 6,251,328

Q2 RESULTS
More parameter-efficient: efficientnet-b0
Parameter difference: 18,179,204
ResNet34 trainable parameters: 3,152,132
ResNet34 trainable percentage: 12.9%


In [7]:
# ==========================================
# Q3: GRAYSCALE INPUT ADAPTATION
# ==========================================

# Strategy A: Duplicate grayscale image into 3 channels
# Strategy B: Collapse RGB convolution weights into 1 channel

# For identical activations:
# W_gray = W_R + W_G + W_B

# Memory comparison
strategy_a_channels = 3
strategy_b_channels = 1

memory_factor = strategy_a_channels / strategy_b_channels

print("Q3 RESULTS")
print("Weight operation: SUM")
print("Strategy A input channels:", strategy_a_channels)
print("Strategy B input channels:", strategy_b_channels)
print("Memory reduction factor:", memory_factor)

print("Q3 Answer: sum, 3")

Q3 RESULTS
Weight operation: SUM
Strategy A input channels: 3
Strategy B input channels: 1
Memory reduction factor: 3.0
Q3 Answer: sum, 3


In [8]:
# ==========================================
# Q4: ATRous CONVOLUTION CALCULATIONS
# ==========================================

kernel_size = 3

def effective_kernel(k, r):
    return k + (k - 1) * (r - 1)

r6 = effective_kernel(kernel_size, 6)
r12 = effective_kernel(kernel_size, 12)

print("Q4 RESULTS")
print("Effective footprint at dilation 6:", r6)
print("Effective footprint at dilation 12:", r12)

advantage = (
    "ASPP captures thin scratches and large diffuse defects "
    "at multiple scales without additional pooling, preserving "
    "fine spatial details."
)

print("Architectural advantage:", advantage)
print(f"Q4 Answer: {r6}, {r12}; {advantage}")

Q4 RESULTS
Effective footprint at dilation 6: 13
Effective footprint at dilation 12: 25
Architectural advantage: ASPP captures thin scratches and large diffuse defects at multiple scales without additional pooling, preserving fine spatial details.
Q4 Answer: 13, 25; ASPP captures thin scratches and large diffuse defects at multiple scales without additional pooling, preserving fine spatial details.


In [9]:
# ==========================================
# Q5: SOFT DICE LOSS
# ==========================================

epsilon = 1.0
total_pixels = 10000
positive_pixels = 100
background_pixels = 9900

# Ground truth
y = np.zeros(total_pixels, dtype=np.float32)
y[:positive_pixels] = 1.0

# Model 1: Uniform prediction
p1 = np.full(total_pixels, 0.01, dtype=np.float32)

# Model 2: Accurate prediction
p2 = np.zeros(total_pixels, dtype=np.float32)
p2[:positive_pixels] = 0.90


def soft_dice_loss(pred, target, eps=1.0):
    intersection = np.sum(pred * target)
    denominator = np.sum(pred) + np.sum(target)

    dice = (2 * intersection + eps) / (denominator + eps)

    return 1 - dice


loss_model1 = soft_dice_loss(p1, y, epsilon)
loss_model2 = soft_dice_loss(p2, y, epsilon)

print("Q5 RESULTS")
print(f"Model 1 Soft Dice Loss: {loss_model1:.4f}")
print(f"Model 2 Soft Dice Loss: {loss_model2:.4f}")

print(f"Q5 Answer: {loss_model1:.4f}, {loss_model2:.4f}")

Q5 RESULTS
Model 1 Soft Dice Loss: 0.9851
Model 2 Soft Dice Loss: 0.0524
Q5 Answer: 0.9851, 0.0524


In [10]:
import torch
import torch.nn.functional as F

EPSILON = 1.0

def soft_dice_loss(pred, target):
    intersection = (pred * target).sum()
    return 1 - (2 * intersection + EPSILON) / (
        pred.sum() + target.sum() + EPSILON
    )

# Completely clean image: 10,000 background pixels
target = torch.zeros(10000)

# Case 1: Perfectly empty prediction
pred_clean = torch.zeros(10000, requires_grad=True)
loss_clean = soft_dice_loss(pred_clean, target)
loss_clean.backward()

print("Clean prediction Dice Loss:", round(loss_clean.item(), 4))
print("Clean prediction gradient:", pred_clean.grad[0].item())

# Case 2: 50 false-positive pixels
pred_noise = torch.zeros(10000)
pred_noise[:50] = 1.0
pred_noise.requires_grad_()

loss_noise = soft_dice_loss(pred_noise, target)
loss_noise.backward()

print("\nFalse-positive Dice Loss:", round(loss_noise.item(), 4))
print("False-positive gradient:", pred_noise.grad[0].item())

# BCE + Dice compound loss
bce = F.binary_cross_entropy(pred_noise.detach().clamp(1e-7, 1-1e-7), target)
compound_loss = 0.5 * bce + 0.5 * loss_noise

print("\nBCE Loss:", round(bce.item(), 4))
print("Compound Loss:", round(compound_loss.item(), 4))

Clean prediction Dice Loss: 0.0
Clean prediction gradient: 1.0

False-positive Dice Loss: 0.9804
False-positive gradient: 0.0003844675375148654

BCE Loss: 0.0797
Compound Loss: 0.5301


In [11]:
import torch

print("Device:", torch.device("cuda" if torch.cuda.is_available() else "cpu"))

print("\nAvailable training histories:")
for name, obj in list(globals().items()):
    if any(k in name.lower() for k in ["history", "scratch", "resnet", "pretrained", "metrics"]):
        if isinstance(obj, (list, dict, tuple)):
            print(f"\n{name}:")
            print(str(obj)[:1500])

print("\nAvailable model objects:")
for name, obj in list(globals().items()):
    if isinstance(obj, torch.nn.Module):
        print(name, "->", type(obj).__name__)

Device: cpu

Available training histories:

Available model objects:
model_a -> Unet
model_b -> Unet


In [12]:
import torch

for name, model in [("model_a", model_a), ("model_b", model_b)]:
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"\n{name}")
    print("Architecture:", model.__class__.__name__)
    print("Total parameters:", f"{total:,}")
    print("Trainable parameters:", f"{trainable:,}")
    print("Encoder:", getattr(model, "encoder", "Not available"))


model_a
Architecture: Unet
Total parameters: 24,430,532
Trainable parameters: 3,152,132
Encoder: ResNetEncoder(
  (conv1): Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2